In [ ]:
!pip install boto3

In [ ]:
import boto3
import pandas as pd
from io import StringIO

# AWSのアクセスキーとシークレットキーを設定

# S3クライアントを作成
s3_client = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)

# S3からCSVファイルを読み込む
response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
csv_content = response["Body"].read().decode("utf-8")

# CSVコンテンツをDataFrameに変換
data = pd.read_csv(StringIO(csv_content))

# データの表示
print(data.head())

In [ ]:
# データの処理（例: 新しい列を追加）
data["new_column"] = data["existing_column"] * 2

# 変更したデータをS3にアップロード
csv_buffer = StringIO()
data.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket=bucket_name, Key="path/to/your/modified_file.csv", Body=csv_buffer.getvalue()
)

In [ ]:
#必要なライブラリのインストール
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client google-drive-api pydrive
     

In [ ]:
# データの読み込み
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import pandas as pd
from io import BytesIO

# Google Drive認証
gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)


# Google Drive上のファイルIDを指定してファイルを取得
file_id = "YOUR_FILE_ID"
file = drive.CreateFile({"id": file_id})
file.GetContentFile("downloaded_file.csv")


# CSVファイルを読み込む
data = pd.read_csv("downloaded_file.csv")


# データの表示
print(data.head())

In [ ]:
# データの処理（例: 新しい列を追加）
data["new_column"] = data["existing_column"] * 2

# 変更したデータをGoogle Driveにアップロード
data.to_csv("modified_file.csv", index=False)
upload_file = drive.CreateFile({"title": "modified_file.csv"})
upload_file.SetContentFile("modified_file.csv")
upload_file.Upload()

print(f'File uploaded with ID: {upload_file["id"]}')

In [ ]:
-- S3 に保存されている CSV ファイルをテーブルとして定義
CREATE EXTERNAL TABLE IF NOT EXISTS my_table (
    column1 STRING,
    column2 INT,
    column3 DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
WITH SERDEPROPERTIES (
    'serialization.format' = ','
)
LOCATION 's3://your-bucket/path/to/csv/'
TBLPROPERTIES ('has_encrypted_data'='false');

-- データを選択して抽出
SELECT
    column1,
    column2,
    column3
FROM
    my_table
WHERE
    column2 > 100
ORDER BY
    column3 DESC;

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# データの読み込み
transaction_history = pd.read_csv("../data/transaction_history.csv", dtype={"customer_id": str})

# purchase_dateを日付型に変換（不正な日付はNaTに変換）
transaction_history["purchase_date"] = pd.to_datetime(
    transaction_history["purchase_date"], format="mixed", errors="coerce"
)

# データの確認
print(transaction_history.head())

  customer_id product_id purchase_date  purchase_amount  redeem_coupon  \
0    00023085   G03257PD    2020-03-13             7200           True   
1    00015469   CKZF21ZC    2023-03-29            15400          False   
2    00015482   7NSE0G0R    2023-12-26            15400          False   
3    00017107   WVCPYZBH    2021-02-19            17100          False   
4    00024156   OELTJ7E2    2020-10-08            24100           True   

   final_settlement_amount payment_method  purchase_channel  
0                   6200.0             現金               2.0  
1                      NaN            カード               2.0  
2                      NaN             現金               1.0  
3                      NaN            カード               2.0  
4                  23100.0            カード               1.0  


In [4]:
# データの分割
train_data = transaction_history[
    (transaction_history["purchase_date"] >= "2018-01-01")
    & (transaction_history["purchase_date"] <= "2021-12-31")
]
test_data = transaction_history[transaction_history["purchase_date"] >= "2022-01-01"]

# Customer_IDごとの累計購入金額の合計を算出
train_data_agg = (
    train_data.groupby("customer_id")["purchase_amount"].sum().reset_index()
)
test_data_agg = test_data.groupby("customer_id")["purchase_amount"].sum().reset_index()

# カラム名の変更
train_data_agg.columns = ["customer_id", "total_purchase_amount"]
test_data_agg.columns = ["customer_id", "total_purchase_amount"]

print(train_data_agg.head())
print(test_data_agg.head())

  customer_id  total_purchase_amount
0    00010000                   9300
1    00010001                   8300
2    00010002                 164700
3    00010003                  46000
4    00010005                 104100
  customer_id  total_purchase_amount
0    00010004                  69300
1    00010006                   8700
2    00010008                  96600
3    00010014                   8500
4    00010018                 117000
